<a href="https://colab.research.google.com/github/thomaslu678/gee-test/blob/main/clean/11_ArcGis_REST_Forest_Inventory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
from datetime import timedelta
import scipy.stats as stats
import rasterio
from rasterio.transform import from_origin
from rasterio.features import rasterize
import geopandas as gpd
from shapely.geometry import Point
import requests
import re



In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import shap

import seaborn as sns

In [4]:
unique_species = np.array(['Zelkova serrata', 'Ulmus species', 'Zelkova  species',
       'Fraxinus americana', 'Fraxinus pennsylvanica', 'VACANT (GENERIC)',
       'vacant site medium', 'Acer negundo', 'Tilia cordata',
       'Acer campestre', 'vacant site small', 'Prunus serrulata',
       'Gleditsia triacanthos', 'Catalpa spp.', 'Crataegus x Lavallei',
       'Malus sylvestris', 'Pyrus calleryana', 'Quercus robur',
       'Acer platanoides', 'Acer saccharum', 'Stump',
       'Acer pseudoplatanus', 'Populus deltoides', 'Ulmus americana',
       'Pinus species', "Prunus virginiana 'Canada Red'",
       'Syringa reticulata', 'Acer x freemanii',
       'Koelreuteria paniculata', "Prunus x yedoensis 'Akebono'",
       "Morus alba 'Fruitless'", 'Aesculus hippocastanum',
       "Ulmus parvifolia 'Frontier'", 'Celtis occidentalis',
       'Celtis reticulata', 'Ailanthus altissima', 'Acer tataricum',
       'Prunus cerasifera', 'Parrotia persica',
       "Picea glauca 'Albertiana'", 'Acer saccharinum', 'Quercus bicolor',
       'vacant site large', 'Acer grandidentatum', 'Cercis canadensis',
       'Tilia tomentosa', "Acer miyabei 'State Street'",
       "Ulmus americana 'Valley Forge'",
       "Acer platanoides 'Crimson King'", 'Acer rubrum',
       'Corylus colurna', 'Ulmus pumila', "Prunus serrulata 'Kwanzan'",
       'Platanus acerifolia', 'Maackia amurensis', 'Populus tremuloides',
       'Juglans regia', 'Pyrus communis', 'Prunus dulcis',
       "Prunus cerasifera 'Krauter Vesuvius'", 'Acer griseum',
       "Crataegus crusgalli 'Inermis'", 'Malus species',
       'Quercus coccinea', 'Pinus nigra',
       "Maclura pomifera 'White Shield'",
       "Salix matsudana 'Globe Navajo'", 'Fraxinus excelsior',
       'Liriodendron tulipifera', "Tilia americana 'Redmond'",
       'Sophora japonica', 'Picea pungens', 'Ginkgo biloba',
       'Cedrus species', 'Rhus typhina', 'Acer species',
       'Cladrastis kentukea', "Aesculus x carnea 'Fort McNair'",
       'Crataegus species', 'Alnus rubra', 'Prunus species',
       "Pyrus calleryana 'Jack'", 'Malus x Spring Snow',
       'Gymnocladus dioicus', 'Prunus armeniaca', 'Pinus edulis',
       'Robinia pseudoacacia', 'Ulmus parvifolia',
       "Gleditsia triacanthos 'Skyline'", "Fraxinus velutina 'Modesto'",
       'Eucommia ulmoides', 'Pyrus spp.', "Pyrus calleryana 'Aristocrat'",
       'Chionanthus retusus', 'Acer ginnala', 'Cedrus atlantica',
       "Pyrus calleryana 'Chanticleer'", 'Populus fremontii',
       'Prunus avium', 'Quercus macrocarpa', 'Juniperus osteosperma',
       'Cotinus coggygria', "Gleditsia triacanthos 'Street Keeper'",
       "Platanus x acerifolia 'Exclamation'",
       "Gleditsia tracanthos 'Imperial'",
       "Fraxinus pennsylvanica 'Patmore'", "Acer platanoides 'Deborah'",
       'Fagus sylvatica', 'Acer palmatum',
       "Platanus acerifolia 'Bloodgood'", 'Prunus yedoensis',
       "Zelkova serrata 'Green Vase'", 'Juglans nigra',
       "Acer negundo 'Sensation'", 'Prunus sargentii', 'Fraxinus species',
       'Lagerstroemia indica', 'Quercus muehlenbergii', 'N/A',
       'Betula species', 'Quercus rubra', 'PLANTING SITE',
       'Taxodium distichum', 'Acer nigrum', 'Salix species',
       'Aesculus x carnea', 'Quercus species', "Ulmus x  'Accolade'",
       'Platycladus orientalis', 'Acer buergerianum', 'Morus spp.',
       'Sorbus spp.', 'Betula papyrifera', 'Crataegus laevigata',
       'Albizia julibrissin', 'Prunus persica', 'Ulmus x hybrid',
       'Pinus ponderosa', 'Syringa species', 'Juniperus  species',
       'Prunus virginiana', "Acer platanoides 'Emerald Queen'",
       'Crataegus phaenopyrum', 'Liquidambar styraciflua',
       'Elaeagnus angustifolia', 'Prunus domestica',
       "Prunus cerasifera 'Thundercloud'", 'Calocedrus decurrens',
       "Ulmus x 'Triumph'", 'Cupressus sempervirens',
       "Pyrus calleryana 'Bradford'", "Populus nigra 'italica'",
       'Acer glabrum', "Acer truncatum 'Pacific Sunset'", 'Picea species',
       'Magnolia species', 'Metasequoia glyptostroboides',
       'Sequoiadendron giganteum', 'Pinus sylvestris',
       "Catalpa bignonioides 'Nana'", 'Fagus species',
       "Zelkova serrata 'Village Green'", "Quercus robur 'Fastigiato'",
       'Amelanchier species', 'Abies  species', 'Quercus gambellii',
       'Tamarix species', 'Ulmus "New Horizon"', "Zelkova 'Wireless'",
       "Koelreuteria paniculata 'Fastigiata'",
       "Pyrus calleryana 'Capital'", "Gleditsia triacanthos 'inermis'",
       'Betula occidentalis', "Ulmus propinqua 'Emerald Sunshine'",
       'Betula pendula', 'Chionanthus virginicus',
       "Fraxinus oxycarpa 'Raywood'", "Pyrus calleryana 'Red Spire'",
       'Laburnum x watereri', 'Carpinus betulus',
       "Salix matsudana 'Corkscrew'", "Populus tremula 'erecta'",
       'Hibiscus species', 'Prunus padus', 'Salix babylonica',
       'Hamamelis japonica', 'Magnolia stellata', 'Picea abies',
       'Juniperus scopulorum', 'Nyssa sylvatica',
       "Aesculus carnea 'Briottii'", 'Quercus alba', 'Chilopsis linearis',
       'Cornus kousa', 'Pinus heldreichii', 'Populus alba',
       'Cedrus deodara', 'Abies concolor', 'Thuja orientalis',
       'Cornus species', 'Pinus monticola', 'Malus x Snowdrift',
       "Robinia pseudoacacia 'Purple Robe'", 'Taxus baccata',
       "Fraxinus americana 'Autumn Purple'", 'Magnolia x soulangiana',
       'Araucaria araucana', "Amelanchier x grandiflora 'Autumn Brill.'",
       "Betula nigra 'Heritage'", "Fraxinus americana 'Autumn Applause'",
       'Abies lasiocarpa', 'Picea engelmannii', 'Pseudotsuga menziesii',
       'Larix decidua', 'Rhus species', 'Malus x Perfect Purple',
       'Robinia neomexican', 'Acer truncatum',
       "Celtis occidentalis 'Chicagoland'", 'Sorbus alnifolia',
       'Rhus trilobata', "Ulmus wilsoniana 'Prospector'",
       'Halesia carolina', 'Populus nigra',
       "Fraxinus pennsylvanica 'Cimmaron'", 'Franklinia alatamaha',
       "Crataegus laevigata 'Pauls Scarlet'", 'Ulmus procera',
       "Acer rubrum 'October Glory'", 'Acer truncatum x platanoides',
       'Quercus paulstris', 'Malus x Red Baron', 'UNKNOWN',
       'Juniperus monosperma', 'Cupressocyparis x leylandii',
       "Syringa pekinensis 'China Snow'", 'Corylus species',
       "Fraxinus angustifolia 'Golden Desert'", 'Maclura pomifera',
       "Fagus sylvatica 'Riversii'", 'Ulmus carpinifolia',
       'Malus x Centurion', "Acer rubrum 'Red Sunset'",
       "Robinia x ambigua 'Idahoenis'", 'Paulownia species',
       "Quercus x robur 'Long'", 'Pinus aristata', 'Aesculus glabra',
       'Fraxinus mandshurica', 'Salix amygdaloides', 'Picea glauca',
       'Tilia americana', 'Alnus rhombifolia', 'Malus ioensis',
       'Thuja occidentalis', 'Chamaecyparis pisifera',
       'Phellodendron amurense', 'Populus angustifolia',
       'Crataegus crus-galli', 'Prunus x yedoensis', 'Euonymus species',
       "Fraxinus pennsylvanica 'Marshalls Seedless'",
       'Cercidiphyllum japonicum', 'OTHER', 'Catalpa speciosa',
       'Crataegus monogyna', 'Populus x canadensis', 'Pinus flexilis',
       "Robinia pseudoacacia 'Frisia'",
       "Fraxinus pennsylvanica 'Johnson'", 'Cydonia oblonga',
       'Pinus thunbergiana', 'Cornus mas', 'Ilex species',
       'Salix discolor', 'Crataegus ambigua', 'Zizyphus jujuba',
       'Sorbus aucuparia', 'Pinus mugo', 'Populus species',
       'Populus balsamifera', "Acer platanoides 'Cleveland'",
       'Prunus blieriana', "Carpinus betulus 'Frans Fontaine'",
       "Robinia pseudoacacia 'Purple Crown'", 'Pinus densiflora',
       'Quercus acutissima', 'Pistacia chinensis', 'Betula nigra',
       'Pinus contorta', 'Juniperus virginiana',
       "Acer platanoides 'Crimson Sentry'", 'Rhamnus cathartica',
       "Fraxinus pennsylvanica 'Summit'", 'Tsuga canadensis',
       'Crataegus douglasii', 'Cercocarapus montanus', 'Picea omorika',
       'Prunus x cistena', 'Ostrya virginiana', 'Chamaecyparis  species',
       'Populus x hybrid', 'Platanus occidentalis', 'Salix bebbiana',
       'Prunus subhirtella', 'Prunus spp.', 'Chitalpa spp.',
       'Styrax japonicus', 'Chamaecyparis thyoides', 'Prunus salicina',
       'Ulmus rubra', 'Quercus shumardii', 'Tilia x europaea',
       'Cedrus libani', 'Chamaecyparis nootkatensis',
       "Populus tremuloides 'Erecta'", 'Pinus wallichiana',
       "Ulmus 'Homestead'", 'Magnolia grandiflora', 'photinia x fraseri',
       'Alnus tenuifolia', "Caragana arborescens 'Pendula'",
       'Celtis laevigata', 'Juglans cinerea', 'Tilia platyphyllos',
       'Carya illinoensis', 'Cotinus obovatus', 'Persea borbonia',
       'Prunus laurocerasus', 'Populus x acuminata', 'Sambucus caerulea',
       'Frangula alnus', 'Salix matsudana',
       "Pinus densiflora 'Umbraculifera'"], dtype=object)

In [7]:
text = """
Table 4. Predicted sizes for 12 species at 15 and 30 years after planting are shown
Crapemyrtle 10.44 16.24 4.79 5.50 3.77 4.68 23.41 41.76
Southern magnolia 18.98 30.19 7.55 9.80 5.80 7.38 65.83 113.87
Ginkgo 12.94 38.77 7.92 11.74 4.98 8.50 45.15 235.29
Sweetgum 23.77 45.77 11.89 15.94 6.60 9.71 152.27 387.07
Goldenrain tree 21.87 37.15 8.77. 10.50 7.17 9.04 72.43 175.21
Camphor tree 26.60 46.05 9.12 12.21 7.38 10.90 86.67 264.01
European ash 24.19 39.12 9.18 10.27 7.54 9.89 93.92 220.47
4.29
Silver maple 27.47 58.12 13.11 18.62 9.22 13.67 151.03 427.56
Chinese pistache 27.15 38.12 10 70 12 41 10 17 1196 179 18 358 48
London plane 38.11 48.17 13.97 15.51 11.48 12.89 262.22 353.87
Zelkova 37.89 53.38 11.36 12.85 12.64 14.91 297.44 589.15
Brodford pear 31 3O 4000 1017 11 1o 10 7 171 U 300
Table 1. Ranges of data used to fit crown-width prediction models for 53 species in the western United States.
Softwood species
Pacific silver fir 218 4 33 5.0 35.8 5 99 2 493 43.16 48.96 –123.95 –120.52 2100 5300 –9 26
White fir 855 3 35 5.0 62.6 10 99 4 445 34.08 46.11 –123.54 –104.95 322 10200 –40 19
Grand fir 610 4 35 5.0 43.9 5 99 12 496 40.54 48.91 –124.33 –114.58 295 6400 –48 20
Corkbark fir 68 5 15 5.2 18.3 45 99 70 222 37.02 41.44 –112.17 –105.15 8200 11200 1 28
Subalpine fir 1262 3 28 5.0 27.4 5 99 3 280 37.44 48.86 –123.05 –105.67 1600 11500 –14 44
California red fir 160 4 36 5.0 52.3 20 99 35 423 33.72 41.28 –123.46 –116.67 4600 9000 –35 26
Shasta red fir 63 6 26 5.0 40.1 40 99 38 288 41.47 43.70 –123.05 –121.60 5500 6600 3 17
Noble fir 50 4 29 5.2 46.0 20 99 85 354 43.94 48.86 –122.35 –119.96 2100 5600 –11 32
Port-Orford cedar 78 3 22 5.0 13.4 15 99 199 270 42.85 43.15 –124.46 –123.88 800 2800 –35 –15
California juniper (w) 28 6 31 5.2 42.6 35 99 47 101 34.82 41.56 –121.03 –118.28 4000 5837 –41 4
Western juniper (w) 302 3 36 5.0 45.3 10 99 8 218 36.35 44.93 –121.91 –113.01 200 9000 –71 27
Utah juniper (w) 402 1 30 5.0 29.7 10 99 8 221 34.32 44.93 –121.91 –104.86 4623 8600 –30 27
Rocky mtn. juniper (w) 144 5 29 5.0 38.1 5 99 5 277 37.02 44.91 –112.46 –103.60 3900 8725 –37 19
One-seed juniper (w) 98 2 28 5.7 36.5 10 99 11 512 37.13 38.53 –105.09 –103.11 4900 6700 –43 –17
Western larch 183 3 28 5.0 23.1 5 99 24 300 44.42 48.97 –121.20 –114.92 295 6400 –25 20
Incense cedar 220 3 32 5.0 42.0 5 99 12 445 35.72 44.71 –123.79 –118.33 1100 6800 –33 11
Engelman spruce 1205 3 29 5.0 35.9 5 99 2 391 37.02 48.89 –123.84 –105.15 295 12000 –25 44
Sitka spruce 53 7 43 5.1 39.2 20 99 12 307 40.54 47.77 –124.46 –123.30 0 2800 –46 –5
Whitebark pine 97 4 29 5.0 24.2 10 99 46 270 37.54 48.62 –123.05 –110.01 6200 10700 6 44
Bristlecone pine 26 7 25 5.0 18.9 65 99 24 121 37.31 39.45 –114.65 –105.06 8410 10700 9 25
Common pinyon (w) 278 5 27 5.0 25.5 30 99 3 500 35.12 40.14 –118.28 –103.32 4600 8725 –40 11
Lodgepole pine 2761 1 40 5.0 62.2 5 99 2 315 36.35 48.90 –124.18 –105.41 200 10900 –23 43
Limber pine 164 3 34 5.0 20.7 5 99 3 256 37.02 45.64 –115.76 –105.15 6400 11200 2 43
Jeffrey pine 108 5 44 5.1 48.1 5 99 12 345 36.10 42.19 –123.75 –118.33 1700 9000 –38 15
Sugar pine 118 5 49 5.0 44.9 15 99 28 445 34.08 42.73 –124.10 –116.94 1100 8200 –47 11
Western white pine 84 4 34 5.1 38.4 15 99 15 423 36.21 48.92 –124.18 –115.87 200 9200 –25 32
Ponderosa pine 1413 1 46 5.0 41.3 5 99 4 414 36.81 48.91 –123.57 –104.07 300 9800 –56 41
Grey pine 37 6 54 5.1 34.8 30 95 9 164 35.70 41.18 –122.90 –118.05 400 4900 –69 –4
Singleleaf pinyon (w) 323 4 30 5.0 22.7 20 99 7 206 34.32 40.26 –119.60 –112.16 200 8900 –71 15
Douglas-fir 4088 1 66 5.0 68.7 5 99 2 696 37.13 48.97 –124.46 –104.95 100 11000 –49 67
Redwood 55 3 31 5.1 35.6 5 99 104 353 37.07 40.68 –124.07 –122.17 400 1800 –55 –39
Western redcedar 439 3 38 5.0 62.0 15 99 13 493 37.90 48.97 –124.42 –109.76 200 8520 –25 49
Western hemlock 1008 4 54 5.0 63.5 5 99 12 496 42.68 48.96 –124.42 –115.93 0 8500 –34 49
Mountain hemlock 209 1 33 5.0 32.4 5 99 28 397 37.64 48.72 –123.55 –115.23 1600 9200 –10 27
Hardwood species
Bigleaf maple 106 8 57 5.1 34.1 10 99 37 405 39.82 48.33 –124.06 –121.38 100 8500 –36 49
Rocky mtn. maple (w) 70 8 39 5.0 26.2 25 99 10 288 40.75 48.64 –123.48 –110.95 1600 7200 –38 14
Bigtooth maple (w) 48 4 20 5.0 14.7 30 99 4 145 37.34 41.13 –112.75 –111.33 5400 8275 –11 11
Red alder 409 3 54 5.0 28.3 10 99 6 386 40.54 48.82 –124.46 –117.30 0 8100 –46 53
White alder 37 8 35 5.1 18.1 25 90 49 270 38.75 45.61 –123.92 –120.98 700 3000 –56 –2
Pacific madrone 164 1 43 5.0 29.7 5 99 13 369 37.21 45.33 –124.29 –121.06 700 6000 –55 15
Curlleaf mtn-mahogany
(w)
227 3 29 5.0 24.0 10 99 10 174 34.32 45.05 –121.91 –109.70 495 9200 –37 27
Tanoak 534 2 41 5.0 31.0 5 99 57 353 37.07 42.52 –124.37 –121.06 300 6000 –55 15
Quaking aspen 1383 1 34 5.0 19.9 5 99 2 256 37.02 48.42 –119.27 –104.07 3200 10800 –9 33
Narrowleaf cottonwood 44 8 35 5.1 16.6 30 90 17 150 37.05 38.85 –108.65 –106.15 5400 8000 –26 –2
Coastal live oak 87 3 53 5.0 40.6 5 99 60 147 34.51 38.62 –122.63 –119.81 500 2000 –73 –54
Canyon live oak 440 5 49 5.0 53.2 5 99 10 696 34.08 42.43 –124.29 –116.94 700 8200 –60 –5
Blue oak 184 5 61 5.0 29.4 15 99 4 172 35.23 40.75 –122.99 –118.55 400 5900 –69 –2
Gambel oak (w) 248 1 19 5.0 15.4 5 99 9 149 37.02 40.16 –113.28 –104.73 6000 8500 –23 11
Oregon white oak 126 6 30 5.0 22.4 15 99 22 200 38.38 45.87 –123.34 –120.75 900 3700 –54 –11
California black oak 239 4 52 5.1 40.3 5 99 20 345 35.37 42.63 –123.65 –118.29 1100 6200 –47 –8
Valley oak 29 5 47 5.1 21.3 10 99 63 172 35.70 39.20 –122.99 –120.96 900 1800 –64 –44
Interior live oak 79 2 37 5.0 19.5 15 99 10 77 35.99 40.70 –123.21 –118.06 600 7300 –60 –5
California laurel 28 10 44 5.1 18.7 30 99 24 271 38.31 43.63 –124.06 –120.39 900 6000 –54 15
Softwood species
Balsam fir (Ahies
halsamia )
Easern redcedar
(Juniperus
virginiana )
Tamarack (Lo-ix
laricina)
Norway spruce
(Picea abies)
White spruce
(P. glauca)
Black spruce
(P. mariana)
Red spruce
(P. rubens )
Jack pine (Pinus
banksiana)
Shortleaf pine
(P. echinata)
Slash pine
(P. elliottii)
Longleaf pine
(P. palustris)
Red pine
(P. resinosa)
Pitch pine
(P. rigida)
Pond pine
(P. serotina )
Eastern white pine
(P. strobus)
Scotch pine
(P. sylvestris)
Loblolly pine
(P. taeda)
Virginia pine
(P. virginiana)
Baldcypress
(Taxodium
distichm )
Northern white cedar 1,238 3 27
( Th ujo
occidentalis )
Eastern hemlock 750 4 40
- (Tsuga ccmrrdensis)
-.-___

Hardwood species
Boxelder 164
(Acer negundo)
Red maple 4,186
(A. rubrum)
Silver maple 133
(A. scharinum)
Sugar maple 2,259
(A. sacharum)
Serviceberry 26
(Amelanchier
arborea)
Yellow birch (Be&a 542
alleghuniensis)
Sweet birch 299
(B. lenta)
River birch 53
(B. nigra)
Paper birch 889
(B. papyrifera)
American horn-beam 95
(Carpinus
cnroliniana)
Bitternut hickory 110
(Cava
cordiformis)
Pignut hickory 284
(C. glabra)
Shagbark hickory (C. 236
ovata)
Black hickory 63
(C. texuna)
Mockernut hickory 275
(C. tomentosa )
Hackberry (C&is 136
occidentalis)
Dogwood (Cornus 152
florida )
Persimmon 64
(Diospyros
virginiana)
American beech 777
(Fagus grundzfolia)
White ash (Fraxinus 701
americana)
Black ash 376
(F. nigra )
Green ash (F. 273
pennsylvanica )
Honeylocust 29
(Gleditsia
triacanthos)
American holly (Rex 100
opaca)
Black walnut 110
(Jugluns~)

Sweetgum 1,115 4 50
(Liquidambar
styraci$ua)
Yellow poplar 1,154 4 61
(Liriodendron
tulipifera)
Cucumber tree 32 10 39
(Magnolia
acuminata)
Sweetbay 117 6 41
(M. virginiana )
Red mulberry (Morus 34 7 46
rubra)
Water tupelo (Nyssa
aquatica)
Blackgum
(N. sylvatica)
Swamp tupelo
(N. sylvatica var.
biflora )
Eastern hophombeam (Ostrya
virginiana)
Sourwood
(Oxydendron
arboreum)
Redbay (Persea
borbonia)
Sycamore (Platanus
occidentalis)
Balsam poplar
(Populus
balsamifera)
Eastern cotton-wood
(P. deltoides)
Bigtooth aspen
(P. grandidentata)
Quaking aspen
(P. tremuloides)
100 8 38
480 6 51
212 4 41
124 7 39
283 5 37
29 3 25
77 11 67
0111 5 26
38 9 80
286 4 44
1,375 5 40
Black cherry (Prunus 737 1 52
serotina)
White oak (Quercus 1,586 5 69
alba)
Scarlet oak 401 5 67
(Q. coccinea)
Northern pin oak (Q. 52 4 44
ellipsoidalis)
Southern red oak (Q. 285 4 57
falcata)
Shingle oak 31 10 55
(Q. imbricaria)
Turkey oak

Laurel oak
(U. rubra)
89 9 55
(Q. laurifolia)
Bur oak 194 2 62
(Q. macrocarpa)
Blackjack oak (Q. 73 3 37
marilandica)
Chinkapin oak (Q. 81 12 46
muehlenbergii)
Water oak 476 4 58
(Q. nigru)
Pin oak 30 12 63
(Q. palustris)
Willow oak 107 9 75
(Q. phellos)
Chestnut oak 997 4 68
(Q. prinus)
Northemredoak(Q. 1,191 2 82
rubra )
Post oak 341 6 45
(Q. stellata)
Black oak 770 5 53
(Q. velutina)
Live oak 36 8 67
(Q. virginiana)
Black locust (Robinia 199 3 48
pseudoacacia)
Sassafras (Sassafrass 218 4 29
albidum)
American bass-wood 456 2 61
(Tilia americana)
Winged elm (Ulmus 70 10 40
alata)
American elm 418 4 99
(U. americana)
Slippery elm
Balsam fir BF 3605 12.2 7.0 1.1 40.1 2.8 1.3 0.3 12.5
Black spruce BS 400 14.6 3.5 4.3 23.6 2.4 0.7 0.8 5.2
Eastern hemlock EH 1127 23.1 11.1 1.3 57.4 5.3 2.0 1.2 12.1
Eastern white pine WP 866 25.6 15.7 1.4 92.2 5.0 2.4 0.6 14.8
Northern white-cedar WC 866 22.7 7.6 1.6 59.7 3.7 1.2 0.8 10.5
Red spruce RS 2994 19.7 7.2 1.2 56.9 3.6 1.4 0.7 12.1
White spruce WS 339 17.4 7.1 1.5 40.6 3.5 1.2 0.6 9.3
Hardwoods
American beech AB 325 21.3 6.7 11.9 43.4 6.0 2.0 1.4 12.2
Gray birch GB 251 6.4 4.7 1.3 25.4 2.1 1.3 0.2 9.4
Northern red oak RO 102 24.3 10.3 12.7 66.8 6.3 2.3 2.1 13.1
Paper birch PB 576 16.4 8.4 1.3 37.3 4.3 2.1 0.2 14.0
Quaking aspen QA 353 18.2 7.5 1.3 45.4 4.2 1.7 0.4 10.7
Red maple RM 1785 18.2 8.6 1.3 54.9 4.9 2.1 0.2 12.2
Sugar maple SM 355 24.4 9.6 12.7 73.9 6.2 2.0 2.4 14.3
Yellow birch YB 388 23.6 8.9 1.5 54.1 6.3 2.2 1.9 13.0
"""

In [16]:
# your inputs
species = unique_species
t = text.lower()

# normalize text
t = re.sub(r"[^a-z\s]", " ", t)
t = re.sub(r"\s+", " ", t)

matches = []
non_matches = []

for s in species:
    s_clean = s.lower()

    # remove cultivar names and quotes
    s_clean = re.sub(r"'[^']*'", "", s_clean)
    s_clean = s_clean.strip()

    parts = s_clean.split()

    genus = parts[0]
    species_epithet = parts[1] if len(parts) > 1 else ""

    full_name = f"{genus} {species_epithet}".strip()

    found = False

    # check full scientific name
    if full_name and full_name in t:
        found = True

    # check genus presence
    elif genus in t:
        found = True

    if found:
        matches.append(s)
    else:
        non_matches.append(s)

print("Matched species:", len(matches))
print("Not matched:", len(non_matches))

print("\nExamples of matches:")
print(matches[:20])

Matched species: 193
Not matched: 141

Examples of matches:
['Zelkova serrata', 'Ulmus species', 'Zelkova  species', 'Fraxinus americana', 'Fraxinus pennsylvanica', 'Acer negundo', 'Tilia cordata', 'Acer campestre', 'Prunus serrulata', 'Gleditsia triacanthos', 'Quercus robur', 'Acer platanoides', 'Acer saccharum', 'Acer pseudoplatanus', 'Populus deltoides', 'Ulmus americana', 'Pinus species', "Prunus virginiana 'Canada Red'", 'Acer x freemanii', "Prunus x yedoensis 'Akebono'"]


# Fetch tree inventory

In [ ]:
url = "https://www.arcgis.com/apps/mapviewer/index.html?webmap=5005408e8d2742eb830b9cee87833d4a"
separator = "webmap="
_, _, id = url.partition(separator)

In [ ]:
def get_operational_layer_urls(portal_base, webmap_id):
    url = f"{portal_base}/sharing/rest/content/items/{webmap_id}/data"
    params = {"f": "pjson"}

    r = requests.get(url, params=params)
    r.raise_for_status()
    data = r.json()

    layers = data.get("operationalLayers", [])

    urls = []
    for layer in layers:
        if "url" in layer:
            urls.append(layer["url"])

    return urls

In [ ]:
def fetch_all_features(layer_url, page_size=2000):
    query_url = f"{layer_url}/query"

    all_features = []
    offset = 0

    while True:
        params = {
            "where": "1=1",
            "outFields": "*",
            "returnGeometry": "true",
            "f": "geojson",
            "resultOffset": offset,
            "resultRecordCount": page_size
        }

        # IMPORTANT: use POST (not GET)
        r = requests.post(query_url, data=params)
        r.raise_for_status()
        data = r.json()

        features = data.get("features", [])

        if not features:
            break

        all_features.extend(features)
        offset += page_size

        print(f"Fetched {len(all_features)} features...")

        # Stop if we got fewer than requested (last page)
        if len(features) < page_size:
            break

    return all_features

In [ ]:
base_url = "https://slcgov.maps.arcgis.com/"

urls = get_operational_layer_urls(base_url, id)

In [ ]:
urls

['https://services.arcgis.com/mMBpeYj0vPFotzbe/arcgis/rest/services/Urban_Forestry_Inventory/FeatureServer/0']

In [ ]:
count_url = f"{urls[0]}/query"

count_params = {
    "where": "1=1",
    "returnCountOnly": "true",
    "f": "json"
}

r = requests.post(count_url, data=count_params)
print("Total features on server:", r.json()["count"])

Total features on server: 110292


In [ ]:
features = fetch_all_features(urls[0])
gdf = gpd.GeoDataFrame.from_features(features)

Fetched 2000 features...
Fetched 4000 features...
Fetched 6000 features...
Fetched 8000 features...
Fetched 10000 features...
Fetched 12000 features...
Fetched 14000 features...
Fetched 16000 features...
Fetched 18000 features...
Fetched 20000 features...
Fetched 22000 features...
Fetched 24000 features...
Fetched 26000 features...
Fetched 28000 features...
Fetched 30000 features...
Fetched 32000 features...
Fetched 34000 features...
Fetched 36000 features...
Fetched 38000 features...
Fetched 40000 features...
Fetched 42000 features...
Fetched 44000 features...
Fetched 46000 features...
Fetched 48000 features...
Fetched 50000 features...
Fetched 52000 features...
Fetched 54000 features...
Fetched 56000 features...
Fetched 58000 features...
Fetched 60000 features...
Fetched 62000 features...
Fetched 64000 features...
Fetched 66000 features...
Fetched 68000 features...
Fetched 70000 features...
Fetched 72000 features...
Fetched 74000 features...
Fetched 76000 features...
Fetched 78000 fe

In [ ]:
gdf.columns

Index(['geometry', 'FID', 'SPP', 'DBH', 'ADDRESS', 'STREET', 'SIDE', 'SITE',
       'Vacant'],
      dtype='object')

In [ ]:
gdf['SPP'].unique()

array(['Zelkova serrata', 'Ulmus species', 'Zelkova  species',
       'Fraxinus americana', 'Fraxinus pennsylvanica', 'VACANT (GENERIC)',
       'vacant site medium', 'Acer negundo', 'Tilia cordata',
       'Acer campestre', 'vacant site small', 'Prunus serrulata',
       'Gleditsia triacanthos', 'Catalpa spp.', 'Crataegus x Lavallei',
       'Malus sylvestris', 'Pyrus calleryana', 'Quercus robur',
       'Acer platanoides', 'Acer saccharum', 'Stump',
       'Acer pseudoplatanus', 'Populus deltoides', 'Ulmus americana',
       'Pinus species', "Prunus virginiana 'Canada Red'",
       'Syringa reticulata', 'Acer x freemanii',
       'Koelreuteria paniculata', "Prunus x yedoensis 'Akebono'",
       "Morus alba 'Fruitless'", 'Aesculus hippocastanum',
       "Ulmus parvifolia 'Frontier'", 'Celtis occidentalis',
       'Celtis reticulata', 'Ailanthus altissima', 'Acer tataricum',
       'Prunus cerasifera', 'Parrotia persica',
       "Picea glauca 'Albertiana'", 'Acer saccharinum', 'Quercus

In [ ]:
gdf = gdf.set_crs("EPSG:4326")

In [ ]:
gdf = gdf.to_crs("EPSG:26912")
pixel_size = 0.6
# Get bounds
minx, miny, maxx, maxy = gdf.total_bounds

# Compute raster dimensions
width = int(np.ceil((maxx - minx) / pixel_size))
height = int(np.ceil((maxy - miny) / pixel_size))

# Define transform (origin = top-left)
transform = from_origin(minx, maxy, pixel_size, pixel_size)

# Prepare shapes for rasterization
shapes = ((geom, 1) for geom in gdf.geometry)

# Rasterize
raster = rasterize(
    shapes=shapes,
    out_shape=(height, width),
    transform=transform,
    fill=0,
    dtype="uint8"
)

/usr/local/lib/python3.12/dist-packages/rasterio/features.py:364: ShapeSkipWarning: Invalid or empty shape None at index 105815 will not be rasterized.
  warnings.warn(


In [ ]:
gdf.iloc[0]

,0
geometry,POINT (423833.0477826344 4512197.0141869)
FID,1
SPP,Zelkova serrata
DBH,10
ADDRESS,533
STREET,S 400 W [400 W]
SIDE,Front
SITE,1
Vacant,


In [ ]:
# Output file
output_path = "points_0_6m_full.tif"

# Write to GeoTIFF
with rasterio.open(
    output_path,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype="uint8",
    crs="EPSG:26912",
    transform=transform,
) as dst:
    dst.write(raster, 1)

print("Raster written to:", output_path)

Raster written to: points_0_6m_full.tif
